# 03 — Evaluate Classification Accuracy

## Purpose

This notebook compares the pipeline-assigned ESCO occupation code with the completed manual coding for the reproducible 200-record validation sample. It reports exact full-code agreement and agreement at progressively shorter ESCO code prefixes, together with error rates by description language and extraction type.

## Inputs

- `classification_validation_sample_200.parquet` ? deterministic sample created by notebook 02.
- `classification_validation_manual_codes.csv` ? completed manual coding for all 200 sampled vacancies.

## Outputs

Six semicolon-delimited CSV files in `output/data-pipeline-answers/validation/`: three supporting accuracy/error summaries and manuscript-ready Tables A3, A4, and A5.

Run from `notebooks/data-pipeline-answers/`.

**Documentation:** [Notebook guide](README.md) · [Data description](../../data/data-pipeline-answers/README.md) · [Output description](../../output/data-pipeline-answers/README.md)


## Interpretation

`full` requires the complete hierarchical ESCO code strings to match. The remaining comparisons use the first 6, 4, 3, 2, or 1 characters of each code. For example, the four-character comparison corresponds to agreement on the leading four-digit occupation group when the code begins with four digits.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# Explicit repository-relative paths are intentional for reviewer-answer analyses.
SAMPLE_FILE = Path(
    "../../data/data-pipeline-answers/interim/"
    "classification_validation_sample_200.parquet"
)
MANUAL_CODES_FILE = Path(
    "../../data/data-pipeline-answers/validation/"
    "classification_validation_manual_codes.csv"
)
OUTPUT_DIR = Path("../../output/data-pipeline-answers/validation")

## 1. Load automated and manual classifications

Both identifiers and code fields are normalised to trimmed strings before merging. One-to-one validation prevents duplicated IDs from silently changing the result.

In [ ]:
if not SAMPLE_FILE.is_file():
    raise FileNotFoundError(f"Validation sample is missing: {SAMPLE_FILE}")
if not MANUAL_CODES_FILE.is_file():
    raise FileNotFoundError(f"Manual coding file is missing: {MANUAL_CODES_FILE}")

classified_data = pd.read_parquet(SAMPLE_FILE)[
    ["id", "esco_code", "desc_lang", "extract_type"]
].rename(columns={"esco_code": "class_code"})
classified_data = classified_data.astype(
    {"id": "string", "class_code": "string"}
)

manual_data = pd.read_csv(
    MANUAL_CODES_FILE,
    sep=";",
    dtype={"id": "string", "esco_code_checked": "string"},
)[["id", "esco_code_checked"]].rename(
    columns={"esco_code_checked": "manual_code"}
)

for dataframe, label in (
    (classified_data, "classification sample"),
    (manual_data, "manual coding"),
):
    if dataframe["id"].duplicated().any():
        raise ValueError(f"Duplicate IDs found in the {label} file.")

classified_data["class_code"] = classified_data["class_code"].str.strip()
manual_data["manual_code"] = manual_data["manual_code"].str.strip()

extra_manual_ids = set(manual_data["id"]) - set(classified_data["id"])
if extra_manual_ids:
    raise ValueError(
        f"Manual coding contains {len(extra_manual_ids)} IDs outside the sample."
    )

joined_data = classified_data.merge(
    manual_data,
    on="id",
    how="left",
    validate="one_to_one",
)
missing_manual_codes = int(joined_data["manual_code"].isna().sum())
if missing_manual_codes:
    raise ValueError(
        f"Manual codes are missing for {missing_manual_codes} sampled records."
    )

print(f"Compared records: {joined_data.shape[0]}")
print(f"Missing manual codes: {missing_manual_codes}")

## 2. Create ESCO code-prefix comparisons

Prefix variables are created from the same normalised full-code strings. No numeric conversion is used, so dots and hierarchical suffixes are preserved.

In [ ]:
code_columns = ["class_code", "manual_code"]
prefix_lengths = [6, 4, 3, 2, 1]

for prefix_length in prefix_lengths:
    for column in code_columns:
        joined_data[f"{column}_{prefix_length}"] = joined_data[
            column
        ].str.slice(0, prefix_length)

## 3. Calculate overall accuracy

Accuracy is the percentage of non-missing automated/manual pairs that match exactly at each prefix length.

In [ ]:
def column_accuracy(
    dataframe: pd.DataFrame,
    reference_column: str,
    comparison_column: str,
) -> float:
    missing_columns = [
        column
        for column in (reference_column, comparison_column)
        if column not in dataframe.columns
    ]
    if missing_columns:
        raise KeyError(f"Missing comparison columns: {missing_columns}")

    valid_pairs = (
        dataframe[reference_column].notna()
        & dataframe[comparison_column].notna()
    )
    if not valid_pairs.any():
        return 0.0
    return float(
        (
            dataframe.loc[valid_pairs, reference_column]
            == dataframe.loc[valid_pairs, comparison_column]
        ).mean()
        * 100
    )


level_pairs = [
    ("full", "class_code", "manual_code"),
    ("6", "class_code_6", "manual_code_6"),
    ("4", "class_code_4", "manual_code_4"),
    ("3", "class_code_3", "manual_code_3"),
    ("2", "class_code_2", "manual_code_2"),
    ("1", "class_code_1", "manual_code_1"),
]

accuracy_by_prefix = pd.DataFrame(
    [
        {
            "esco_prefix": level,
            "accuracy_percent": round(
                column_accuracy(joined_data, automated, manual), 2
            ),
        }
        for level, automated, manual in level_pairs
    ]
)
accuracy_by_prefix

## 4. Calculate grouped error rates

Within each language or extraction-type group, the error rate is `100 ? (compared ? correct) / compared`. The published pivot tables retain one row per group and one column per ESCO prefix comparison.

In [ ]:
def classification_error_rates(
    dataframe: pd.DataFrame,
    group_column: str,
) -> pd.DataFrame:
    if group_column not in dataframe.columns:
        raise KeyError(f"Grouping column is missing: {group_column}")

    rows = []
    for level, automated_column, manual_column in level_pairs:
        for group_value, group in dataframe.groupby(
            group_column, dropna=False
        ):
            valid_pairs = (
                group[automated_column].notna()
                & group[manual_column].notna()
            )
            compared = int(valid_pairs.sum())
            correct = int(
                (
                    group.loc[valid_pairs, automated_column]
                    == group.loc[valid_pairs, manual_column]
                ).sum()
            )
            errors = compared - correct
            error_percent = (
                round(errors / compared * 100, 2)
                if compared
                else np.nan
            )
            rows.append(
                {
                    group_column: group_value,
                    "esco_prefix": level,
                    "compared": compared,
                    "errors": errors,
                    "error_percent": error_percent,
                }
            )

    long_result = pd.DataFrame(rows)
    pivot = long_result.pivot(
        index=group_column,
        columns="esco_prefix",
        values="error_percent",
    )
    pivot.index.name = group_column
    return pivot


error_rate_by_language = classification_error_rates(
    joined_data, "desc_lang"
)
error_rate_by_extract_type = classification_error_rates(
    joined_data, "extract_type"
)

error_rate_by_language

## 5. Save validation results

The accuracy table is written without an artificial row-index column. Grouped error tables retain their meaningful language or extraction-type index.

In [ ]:
level_descriptions = {
    "1": "Major group",
    "2": "Sub-major group",
    "3": "Minor group",
    "4": "Unit group",
    "6": "ESCO sub-unit",
    "full": "Most granular ESCO code",
}
manuscript_level_order = ["1", "2", "3", "4", "6", "full"]
table_a3 = accuracy_by_prefix.set_index("esco_prefix").reindex(
    manuscript_level_order
).reset_index()
table_a3.insert(
    1, "Description", table_a3["esco_prefix"].map(level_descriptions)
)
table_a3 = table_a3.rename(
    columns={"esco_prefix": "ESCO level", "accuracy_percent": "Accuracy (%)"}
)
table_a3["ESCO level"] = table_a3["ESCO level"].replace({"full": "Full code"})

overall_error = (
    100 - accuracy_by_prefix.set_index("esco_prefix")["accuracy_percent"]
).round(2)

def build_manuscript_error_table(
    pivot: pd.DataFrame,
    group_column: str,
    output_label: str,
    group_order,
    group_labels,
) -> pd.DataFrame:
    ordered = pivot.reindex(group_order)[manuscript_level_order].copy()
    group_sizes = joined_data[group_column].value_counts().reindex(group_order).astype(int)
    ordered.insert(0, "N", group_sizes)
    ordered.insert(0, output_label, [group_labels[value] for value in group_order])
    ordered = ordered.reset_index(drop=True).rename(
        columns={
            "1": "Level 1", "2": "Level 2", "3": "Level 3",
            "4": "Level 4", "6": "Level 6", "full": "Full code",
        }
    )
    weighted_row = {
        output_label: "Weighted average", "N": len(joined_data),
        "Level 1": overall_error.loc["1"],
        "Level 2": overall_error.loc["2"],
        "Level 3": overall_error.loc["3"],
        "Level 4": overall_error.loc["4"],
        "Level 6": overall_error.loc["6"],
        "Full code": overall_error.loc["full"],
    }
    return pd.concat([ordered, pd.DataFrame([weighted_row])], ignore_index=True)

table_a4 = build_manuscript_error_table(
    error_rate_by_language, "desc_lang", "Language",
    ["en", "uk", "ru", "cs", "pl"],
    {"en": "English", "uk": "Ukrainian", "ru": "Russian", "cs": "Czech", "pl": "Polish"},
)
table_a5 = build_manuscript_error_table(
    error_rate_by_extract_type, "extract_type", "Matching method",
    ["preferredLabel", "preferredLabel_fuzzy", "altLabels_fuzzy", "altLabels", "esco_code_similarity"],
    {
        "preferredLabel": "Preferred label",
        "preferredLabel_fuzzy": "Preferred label fuzzy",
        "altLabels_fuzzy": "Alternative label fuzzy",
        "altLabels": "Alternative label",
        "esco_code_similarity": "esco code similarity",
    },
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

accuracy_by_prefix.to_csv(
    OUTPUT_DIR / "classification_accuracy_by_esco_prefix.csv",
    index=False,
    sep=";",
)
error_rate_by_language.to_csv(
    OUTPUT_DIR / "classification_error_rate_by_language.csv",
    sep=";",
)
error_rate_by_extract_type.to_csv(
    OUTPUT_DIR / "classification_error_rate_by_extract_type.csv",
    sep=";",
)
table_a3.to_csv(
    OUTPUT_DIR / "table_a3_manual_validation_accuracy.csv", index=False, sep=";"
)
table_a4.to_csv(
    OUTPUT_DIR / "table_a4_error_rates_by_language.csv", index=False, sep=";"
)
table_a5.to_csv(
    OUTPUT_DIR / "table_a5_error_rates_by_matching_method.csv", index=False, sep=";"
)

print(f"Saved validation results: {OUTPUT_DIR}")

## Expected overall accuracy

| ESCO comparison | Accuracy |
|---|---:|
| Full code | 39.0% |
| First 6 characters | 58.5% |
| First 4 characters | 80.0% |
| First 3 characters | 82.5% |
| First 2 characters | 88.0% |
| First 1 character | 91.0% |